In [1]:
import os
import json
import zlib
import copy
import time
import pymongo
import requests
from util.utility import get_soup

In [2]:
myclient = pymongo.MongoClient('mongodb://localhost:27017/')
mycol = myclient['local']['MyGames']

In [ ]:
new_games = list(mycol.find({}).sort({'_id': -1}).limit(50))
for game_obj in new_games:
    print(game_obj['title'], '#', game_obj.get('Release Date', [None])[0], '#',
          game_obj.get('metacritics-critics', '/'), '#',
          game_obj.get('metacritics-users', '/'), '#',
          game_obj.get('steam-positive', '/'), '#',
          game_obj.get('steam-nb-users', '/'), '#',
          game_obj.get('hltb-main', '/'), '#',
          game_obj.get('hltb-main+', '/'), '#',
          game_obj.get('hltb-complete', '/')
    )

In [ ]:
with open('Platforms.txt', 'r', encoding='utf-8') as file:
    platforms = file.readlines()

for line in platforms:
    line = line.replace('\n', '')
    split_line = line.split('\t')
    title, title_platform = split_line[0], split_line[1]
    franchise = '/'
    if len(split_line) > 2 and split_line[2] not in ['None', 'none', 'N/A', 'n/a', '']:
        franchise = split_line[2]
    print(title, title_platform, franchise)
    mycol.update_one({'title': title}, {'$set': {'platform': title_platform, 'franchise': franchise}})

In [3]:
with open ('Genres.txt', 'r', encoding='utf-8') as file:
    accepted_genres = file.readlines()

genres_dict = dict()
for line in accepted_genres:
    line = line.replace('\n', '')
    split_line = line.split('\t')
    orig_genre, genre_list = split_line[0], split_line[1].split(' # ')
    genres_dict[orig_genre] = genre_list

In [4]:
games = list(mycol.find({}, {'Genres': 1, 'title': 1}).sort({'_id': -1}))
for game_obj in games:
    new_genres = set()
    for genre in game_obj.get('Genres', []):
        if genre in genres_dict and genres_dict[genre][0] != '/':
            new_genres.update({g for g in genres_dict[genre]})
    
    mycol.update_one({'title': game_obj['title']}, {'$set': {'Top Genres': sorted(new_genres)}})

In [ ]:
all_games_titles = list(mycol.find({}, {'_id': 1, 'title': 1, 'Similar Games': 1, 'igdb-similar_games': 1, 'giantbomb-similar-titles': 1, 'giantbomb-name': 1}))
all_titles = {game['title'] for game in all_games_titles}
all_gb_titles = {game['giantbomb-name'] for game in all_games_titles if 'giantbomb-name' in game}

for game_obj in all_games_titles:
    similar_games = set()

    for similar_game in game_obj.get('igdb-similar_games', '').split('; '):
        if similar_game == '':
            continue
        if similar_game in all_gb_titles or similar_game in all_titles:
            similar_games.add(similar_game)

    for similar_game in game_obj.get('giantbomb-similar-titles', '').split('; '):
        if similar_game == '':
            continue
        if similar_game in all_gb_titles or similar_game in all_titles:
            similar_games.add(similar_game)

    if len(similar_games) > 0:
        mycol.update_one({'_id': game_obj['_id']}, {'$set': {'Similar Games': sorted(similar_games)}})
        print(game_obj['title'], similar_games)